# 05 - Consolidação

Gera Excel consolidado e relatórios de auditoria a partir dos YAMLs aprovados.

**Outputs:**
- `saec_consolidated_YYYYMMDD.xlsx` - Planilha com todas as extrações
- `audit_summary_YYYYMMDD.csv` - Relatório de validação

## 1. Setup

In [ ]:
import sys
from pathlib import Path
from datetime import datetime
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
SYSTEM_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
SRC_DIR = SYSTEM_DIR / 'src'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from config import paths
from mapping_sync import sync_mapping_with_validation_and_qa
from consolidate import consolidate_yamls, generate_statistics, print_statistics

print('Modules loaded')


# Sync mapping (valid + QA OK)
DRY_RUN = True
qa_reports = sorted(paths.CONSOLIDATED.glob('qa_report_*.csv'))
if not qa_reports:
    raise SystemExit('Nenhum qa_report encontrado em outputs/consolidated')
qa_path = qa_reports[-1]

preview = sync_mapping_with_validation_and_qa(
    mapping_path=paths.MAPPING_CSV,
    yamls_dir=paths.YAMLS,
    qa_report_path=qa_path,
    dry_run=DRY_RUN,
)
print(preview.head())


## 2. Verificar YAMLs Disponíveis

In [ ]:
yamls = sorted(paths.YAMLS.glob("*.yaml"))

print(f" YAMLs aprovados: {len(yamls)}")
print(f"   Diretório: {paths.YAMLS}")

if yamls:
    print("\n   Arquivos:")
    for y in yamls[:10]:
        print(f"   - {y.name}")
    if len(yamls) > 10:
        print(f"   ... e mais {len(yamls) - 10}")
else:
    print("\n Nenhum YAML encontrado!")
    print("   Execute 03_Extracao_LLM.ipynb primeiro")

## 3. Consolidar em Excel

In [ ]:
# Gerar nomes de arquivo com timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
excel_path = paths.CONSOLIDATED / f"saec_consolidated_{timestamp}.xlsx"
audit_path = paths.CONSOLIDATED / f"audit_summary_{timestamp}.csv"

print(f" Consolidando {len(yamls)} YAMLs...")
print(f"   Excel: {excel_path.name}")
print(f"   Auditoria: {audit_path.name}")

In [ ]:
# Executar consolidação
df = consolidate_yamls(
    yamls_dir=paths.YAMLS,
    output_excel=excel_path,
    output_audit=audit_path,
    only_valid=True  # Só inclui YAMLs que passam validação
)

## 4. Preview dos Dados

In [ ]:
if not df.empty:
    # Colunas principais para preview
    cols_preview = [
        "ArtigoID", "Ano", "Referência_Curta",
        "SegmentoO&G", "ClasseIA", "Maturidade", "ResultadoTipo"
    ]
    
    # Filtrar colunas que existem
    cols_exist = [c for c in cols_preview if c in df.columns]
    
    print(f" Preview ({len(df)} artigos):")
    df[cols_exist].head(10)
else:
    print("Nenhum dado consolidado")

## 5. Estatísticas Descritivas

In [ ]:
if not df.empty:
    stats = generate_statistics(df)
    print_statistics(stats)
else:
    print("Nenhum dado para estatísticas")

## 6. Visualizações

In [ ]:
if not df.empty:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Segmento O&G
    if "SegmentoO&G" in df.columns:
        df["SegmentoO&G"].value_counts().plot(
            kind="bar", ax=axes[0, 0], color="steelblue"
        )
        axes[0, 0].set_title("Distribuição por Segmento O&G")
        axes[0, 0].tick_params(axis='x', rotation=45)
    
    # Classe IA
    if "ClasseIA" in df.columns:
        df["ClasseIA"].value_counts().plot(
            kind="bar", ax=axes[0, 1], color="coral"
        )
        axes[0, 1].set_title("Distribuição por Classe de IA")
        axes[0, 1].tick_params(axis='x', rotation=45)
    
    # Maturidade
    if "Maturidade" in df.columns:
        df["Maturidade"].value_counts().plot(
            kind="pie", ax=axes[1, 0], autopct='%1.1f%%'
        )
        axes[1, 0].set_title("Distribuição por Maturidade")
    
    # Anos
    if "Ano" in df.columns:
        df["Ano"].value_counts().sort_index().plot(
            kind="bar", ax=axes[1, 1], color="seagreen"
        )
        axes[1, 1].set_title("Distribuição por Ano")
    
    plt.tight_layout()
    
    # Salvar figura
    fig_path = paths.CONSOLIDATED / f"statistics_{timestamp}.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"\n Figura salva: {fig_path.name}")
    
    plt.show()
else:
    print("Nenhum dado para visualizar")

## 7. Verificar Arquivos Gerados

In [ ]:
print(" Arquivos gerados:")

for f in sorted(paths.CONSOLIDATED.glob("*")):
    size_kb = f.stat().st_size / 1024
    print(f"   {f.name} ({size_kb:.1f} KB)")

## 8. Relatório Final

In [ ]:
print("=" * 60)
print("SAEC-O&G - RELATÓRIO DE CONSOLIDAÇÃO")
print("=" * 60)
print(f"\nData: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Versão do Guia: v3.3")
print(f"\n Métricas:")
print(f"   Total de artigos: {len(df) if not df.empty else 0}")
print(f"   YAMLs processados: {len(yamls)}")

print(f"\n Arquivos de saída:")
print(f"   Excel: {excel_path}")
print(f"   Auditoria: {audit_path}")

print("\n" + "=" * 60)
print(" Consolidação completa!")
print("=" * 60)

## 9. Abrir Arquivos (Opcional)

In [ ]:
# Descomentar para abrir o Excel automaticamente
# import os
# os.startfile(str(excel_path))  # Windows